# Module 08 — Notebook 03: Boundary Detection Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_08_RL_For_Image_Segmentation/03_Boundary_Detection_Agent/boundary_detection_agent.ipynb)

---

## Learning Objectives

1. Formulate boundary tracing as a sequential decision problem (MDP).
2. Design a state that encodes position, local gradient features, and direction history.
3. Implement a Q-learning agent with 8 directional actions.
4. Train the agent to trace object boundaries on synthetic images.
5. Visualize the agent's path overlaid on the image and analyse training progress.

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for segmentation (TINY - under 200MB total)
import torchvision
import numpy as np

# CIFAR-10: 60,000 real photos (already cached if downloaded before)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = [np.array(cifar10[i][0]) for i in range(500)]
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded (32x32 RGB)")

# FashionMNIST: 60,000 real clothing images (30MB only!)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (28x28)")
print("   Classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Boot")

# Generate segmentation masks from CIFAR-10 using simple thresholding
# (This gives us real images with automatic pseudo-masks - no large dataset needed!)
def generate_pseudo_masks(images, threshold=128):
    """Generate binary masks from real images using Otsu-like thresholding."""
    masks = []
    for img in images:
        gray = np.mean(img, axis=2)
        mask = (gray > np.median(gray)).astype(np.uint8)
        masks.append(mask)
    return masks

pseudo_masks = generate_pseudo_masks(real_images)
print(f"✅ Generated {len(pseudo_masks)} segmentation masks from real images")
print("   Total download: ~200MB (CIFAR-10 + FashionMNIST)")

## Deep Derivation: Edge Detection Theory and Boundary Optimization

### Step 1: Image Gradient — First-Order Edge Detection
$$\nabla I(x,y) = \begin{bmatrix} \frac{\partial I}{\partial x} \\ \frac{\partial I}{\partial y} \end{bmatrix}$$

**Gradient magnitude:** $|\nabla I| = \sqrt{\left(\frac{\partial I}{\partial x}\right)^2 + \left(\frac{\partial I}{\partial y}\right)^2}$

**Gradient direction:** $\theta = \arctan\left(\frac{\partial I / \partial y}{\partial I / \partial x}\right)$

### Step 2: Canny Edge Detector — Optimality Conditions
Canny defined three criteria for an optimal edge detector:

**1. Detection:** Maximize SNR: $\text{SNR} = \frac{|\int_{-\infty}^{0} f(x) dx|}{\sigma_n \sqrt{\int_{-\infty}^{0} f^2(x) dx}}$

**2. Localization:** Minimize distance to true edge: $\text{Loc} = \frac{|f'(0)|}{\sigma_n \sqrt{\int_{-\infty}^{0} f'^2(x) dx}}$

**3. Single response:** One detection per edge.

**Proof the optimal filter is approximately Gaussian derivative:**

Maximizing $\Sigma = \text{SNR} \cdot \text{Loc}$ via calculus of variations gives Euler-Lagrange equation:
$$2f''(x) - 2\lambda_1 f(x) + 2\lambda_2 f''(x) = 0$$

The solution is $f(x) \approx -\frac{d}{dx}G_\sigma(x) = \frac{x}{\sigma^2}G_\sigma(x) \quad \blacksquare$

### Step 3: Non-Maximum Suppression (NMS) — Thinning Edges
For each pixel $(i,j)$ with gradient direction $\theta$:
$$\text{NMS}(i,j) = \begin{cases} |\nabla I(i,j)| & \text{if } |\nabla I(i,j)| \geq |\nabla I_{\theta^+}| \text{ and } |\nabla I(i,j)| \geq |\nabla I_{\theta^-}| \\ 0 & \text{otherwise} \end{cases}$$

where $\theta^+, \theta^-$ are the neighbors along the gradient direction.

**Proof NMS produces 1-pixel-wide edges:**

After NMS, no two adjacent pixels along the gradient direction both have non-zero values — only the local maximum survives. The gradient direction is perpendicular to the edge, so edges are exactly 1 pixel wide. $\blacksquare$

### Step 4: Hysteresis Thresholding — Connected Edge Linking
Two thresholds $T_{\text{low}} < T_{\text{high}}$:
- **Strong edge:** $|\nabla I| > T_{\text{high}}$ (definitely an edge)
- **Weak edge:** $T_{\text{low}} < |\nabla I| < T_{\text{high}}$ (edge only if connected to strong edge)
- **Non-edge:** $|\nabla I| < T_{\text{low}}$ (discarded)

**Proof hysteresis reduces false positives:**

A weak pixel is accepted only if $\exists$ path of weak pixels connecting it to a strong pixel. Isolated noise pixels (weak but unconnected) are rejected — this is a connected component constraint. $\blacksquare$

### Step 5: Boundary F-Measure (BF Score)
$$\text{BF} = \frac{2 \cdot P_b \cdot R_b}{P_b + R_b}$$

where boundary precision $P_b = \frac{\text{matched predicted boundary pixels}}{\text{total predicted boundary pixels}}$ and boundary recall $R_b = \frac{\text{matched GT boundary pixels}}{\text{total GT boundary pixels}}$

Matching uses distance tolerance $\theta_d$ (typically 2 pixels):
$$\text{match}(p, B_{\text{GT}}) = \mathbb{1}\left[\min_{q \in B_{\text{GT}}} \|p - q\|_2 \leq \theta_d\right]$$

### Step 6: Active Contours (Snakes) — Energy Minimization
$$E_{\text{snake}} = \int_0^1 \left[\alpha|\mathbf{v}'(s)|^2 + \beta|\mathbf{v}''(s)|^2 + E_{\text{ext}}(\mathbf{v}(s))\right] ds$$

- $\alpha|\mathbf{v}'|^2$: elasticity (resists stretching)
- $\beta|\mathbf{v}''|^2$: rigidity (resists bending)
- $E_{\text{ext}}$: image force (attracts to edges): $E_{\text{ext}} = -|\nabla I|^2$

**Euler-Lagrange equation:**
$$\alpha \mathbf{v}''(s) - \beta \mathbf{v}''''(s) - \nabla E_{\text{ext}} = 0$$

**Proof this equation gives the energy minimum:**

By calculus of variations, setting $\frac{\delta E}{\delta \mathbf{v}} = 0$ and integrating by parts twice yields the Euler-Lagrange equation above. $\blacksquare$

### Step 7: RL for Boundary Detection — Tracing Agent
**State:** current position on boundary + local image patch + boundary traced so far

**Actions:** $a_t \in \{$move in 8 directions, mark as boundary, skip$\}$

**Reward:**
$$r_t = \begin{cases} +1 & \text{if marked pixel is within } \theta_d \text{ of GT boundary} \\ -1 & \text{if marked pixel is far from GT boundary} \\ -0.1 & \text{step penalty (encourages efficiency)} \end{cases}$$

**Connection to RL:** The agent learns to TRACE boundaries like a human annotator — following edges while skipping noise. The sequential nature of boundary tracing maps naturally to the MDP framework, with the agent's policy learning local edge-following rules that produce globally coherent boundaries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from collections import defaultdict
from skimage.filters import sobel
import random
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

print('All imports successful.')

---
## 1. Mathematical Formulation: MDP for Boundary Tracing

We formulate boundary tracing as an MDP $\mathcal{M} = (\mathcal{S}, \mathcal{A}, R, T, \gamma)$.

### State Space $\mathcal{S}$

The agent state at time $t$ is:

$$
s_t = \Big(x_t,\; y_t,\; \{g(x_t+\delta_r, y_t+\delta_c)\}_{\delta_r,\delta_c \in \{-1,0,1\}},\; d_{t-1}\Big)
$$

where:
- $(x_t, y_t)$ is the agent's current pixel position,
- $g(\cdot)$ is the **gradient magnitude** (Sobel edge map) quantised to discrete bins,
- $d_{t-1} \in \{0,\ldots,7\}$ is the direction of the previous move.

The local $3 \times 3$ gradient patch is flattened and discretised:

$$
\phi(s_t) = \Big(\text{bin}(g_{-1,-1}), \ldots, \text{bin}(g_{1,1}),\; d_{t-1}\Big) \in \mathbb{Z}^{10}
$$

### Action Space $\mathcal{A}$

Eight compass directions:

$$
\mathcal{A} = \{\text{N}, \text{NE}, \text{E}, \text{SE}, \text{S}, \text{SW}, \text{W}, \text{NW}\}
$$

Movement offsets:

$$
\Delta(a) = \{(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1),(-1,-1)\}
$$

### Reward Function $R$

$$
r_t = \begin{cases}
+1 & \text{if } g(x_{t+1}, y_{t+1}) \ge \tau \;\text{(on edge)} \\
-1 & \text{if } g(x_{t+1}, y_{t+1}) < \tau \;\text{(off edge)} \\
+5 & \text{if agent returns to start (loop bonus)} \\
-2 & \text{if agent goes out of bounds}
\end{cases}
$$

where $\tau$ is an edge threshold.

### Transition $T$

Deterministic: $(x_{t+1}, y_{t+1}) = (x_t, y_t) + \Delta(a_t)$.

### Value Function (Q-learning)

$$
Q(s,a) \leftarrow Q(s,a) + \alpha\big[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big]
$$

---
## 2. Synthetic Image with Clear Boundaries

In [ ]:
def create_boundary_image(size=64):
    """Create a synthetic image with a circle — the boundary is well-defined."""
    img = np.zeros((size, size), dtype=np.float32)
    cy, cx, radius = size // 2, size // 2, size // 4
    yy, xx = np.ogrid[:size, :size]
    circle_mask = (yy - cy)**2 + (xx - cx)**2 <= radius**2
    img[circle_mask] = 1.0
    img += np.random.normal(0, 0.02, img.shape).astype(np.float32)
    img = np.clip(img, 0, 1)

    edge_map = sobel(img)
    edge_map = edge_map / (edge_map.max() + 1e-8)

    boundary_mask = edge_map > 0.3
    return img, edge_map, boundary_mask, (cy, cx, radius)

IMG_SIZE = 64
image, edge_map, boundary_mask, circle_params = create_boundary_image(IMG_SIZE)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Synthetic Image'); axes[0].axis('off')
axes[1].imshow(edge_map, cmap='hot'); axes[1].set_title('Sobel Edge Map'); axes[1].axis('off')
axes[2].imshow(boundary_mask, cmap='gray'); axes[2].set_title('Binary Boundary'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## Deep Derivation: Active Contour (Snake) Energy Minimization

### Step 1: Snake Energy Functional

An active contour $\mathbf{v}(s) = (x(s), y(s))$ for $s \in [0, 1]$ minimizes:

$$E_{\text{snake}} = \int_0^1 \left[\alpha |\mathbf{v}'(s)|^2 + \beta |\mathbf{v}''(s)|^2 + E_{\text{ext}}(\mathbf{v}(s))\right] ds$$

**Physical interpretation:**
- $\alpha |\mathbf{v}'(s)|^2$: **Elasticity** — penalizes stretching (first derivative = tangent vector magnitude)
- $\beta |\mathbf{v}''(s)|^2$: **Rigidity** — penalizes bending (second derivative = curvature)
- $E_{\text{ext}}$: **External force** — attracts the contour to image edges

### Step 2: Euler-Lagrange Equation for Optimal Contour

To minimize $E_{\text{snake}}$, we apply the calculus of variations. For a functional $J[\mathbf{v}] = \int F(s, \mathbf{v}, \mathbf{v}', \mathbf{v}'') \, ds$, the necessary condition $\frac{\delta J}{\delta \mathbf{v}} = 0$ gives:

$$\frac{\partial F}{\partial \mathbf{v}} - \frac{d}{ds}\frac{\partial F}{\partial \mathbf{v}'} + \frac{d^2}{ds^2}\frac{\partial F}{\partial \mathbf{v}''} = 0$$

**Computing each term:**

1. $\frac{\partial F}{\partial \mathbf{v}} = \nabla E_{\text{ext}}(\mathbf{v})$

2. $\frac{d}{ds}\frac{\partial F}{\partial \mathbf{v}'} = \frac{d}{ds}(2\alpha \mathbf{v}') = 2\alpha \mathbf{v}''$

3. $\frac{d^2}{ds^2}\frac{\partial F}{\partial \mathbf{v}''} = \frac{d^2}{ds^2}(2\beta \mathbf{v}'') = 2\beta \mathbf{v}''''$

**Euler-Lagrange equation:**

$$-\alpha \mathbf{v}''(s) + \beta \mathbf{v}''''(s) + \nabla E_{\text{ext}}(\mathbf{v}(s)) = 0$$

This is a fourth-order PDE. In discrete form with $N$ control points, it becomes a linear system. $\blacksquare$

### Step 3: Discrete Snake — Matrix Formulation

Discretizing at $N$ points: $\mathbf{v}_i = \mathbf{v}(i/N)$, approximate derivatives by finite differences:

$$\mathbf{v}' \approx \mathbf{v}_{i+1} - \mathbf{v}_i, \quad \mathbf{v}'' \approx \mathbf{v}_{i+1} - 2\mathbf{v}_i + \mathbf{v}_{i-1}$$

The internal energy becomes a quadratic form:

$$E_{\text{int}} = \mathbf{v}^T (\alpha \mathbf{A} + \beta \mathbf{B}) \mathbf{v} = \mathbf{v}^T \mathbf{K} \mathbf{v}$$

where $\mathbf{A}$ and $\mathbf{B}$ are banded matrices (pentadiagonal for $\mathbf{B}$).

**Iterative solution:** $\mathbf{v}^{(t+1)} = (\mathbf{K} + \tau \mathbf{I})^{-1}(\tau \mathbf{v}^{(t)} - \nabla E_{\text{ext}})$

### Step 4: External Energy — Edge Attraction

The standard choice: $E_{\text{ext}}(\mathbf{v}) = -|\nabla(G_\sigma * I)|^2$

**Gradient Vector Flow (GVF)** improves capture range by diffusing edge information:

$$\min_{\mathbf{u}} \int \mu |\nabla \mathbf{u}|^2 + |\nabla f|^2 |\mathbf{u} - \nabla f|^2 \, dx \, dy$$

where $f = |\nabla(G_\sigma * I)|$. Near edges ($|\nabla f|$ large), $\mathbf{u} \approx \nabla f$. Far from edges ($|\nabla f|$ small), $\mathbf{u}$ is smooth (diffused). $\blacksquare$

### Step 5: Connection to RL Boundary Tracing

The active contour minimizes energy globally, while our RL agent makes **local decisions** (which direction to move). The connection:

- The snake's external energy $E_{\text{ext}}$ corresponds to the RL agent's **immediate reward** (on-edge = high reward)
- The snake's elasticity $\alpha|\mathbf{v}'|^2$ corresponds to the agent's **preference for smooth paths** (implicitly learned)
- The snake's rigidity $\beta|\mathbf{v}''|^2$ corresponds to **penalizing sharp turns** (can be added as reward shaping)

**Key advantage of RL:** The agent can handle topology changes (splitting/merging contours) that classical snakes cannot, since the agent doesn't assume a fixed contour parameterization.

---
## 3. Boundary Tracing Environment

In [ ]:
DIRECTIONS = [(-1,0), (-1,1), (0,1), (1,1), (1,0), (1,-1), (0,-1), (-1,-1)]
DIR_NAMES = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']

class BoundaryTracingEnv:
    def __init__(self, image, edge_map, boundary_mask, edge_threshold=0.3, max_steps=300):
        self.image = image
        self.edge_map = edge_map
        self.boundary_mask = boundary_mask
        self.h, self.w = image.shape
        self.threshold = edge_threshold
        self.max_steps = max_steps
        self.num_bins = 5

    def reset(self):
        boundary_pixels = np.argwhere(self.boundary_mask)
        idx = np.random.randint(len(boundary_pixels))
        self.start_pos = tuple(boundary_pixels[idx])
        self.pos = self.start_pos
        self.prev_dir = 0
        self.path = [self.pos]
        self.visited = {self.pos}
        self.steps = 0
        return self._get_state()

    def _get_state(self):
        r, c = self.pos
        patch = []
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < self.h and 0 <= nc < self.w:
                    patch.append(self.edge_map[nr, nc])
                else:
                    patch.append(0.0)
        binned = tuple(np.digitize(patch, bins=np.linspace(0, 1, self.num_bins)))
        return binned + (self.prev_dir,)

    def step(self, action):
        self.steps += 1
        dr, dc = DIRECTIONS[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc

        if not (0 <= nr < self.h and 0 <= nc < self.w):
            return self._get_state(), -2.0, True

        self.pos = (nr, nc)
        self.prev_dir = action
        self.path.append(self.pos)

        on_edge = self.edge_map[nr, nc] >= self.threshold

        # Loop detection
        dist_to_start = abs(nr - self.start_pos[0]) + abs(nc - self.start_pos[1])
        if dist_to_start <= 1 and self.steps > 10:
            return self._get_state(), 5.0, True

        if self.steps >= self.max_steps:
            return self._get_state(), 0.0, True

        reward = 1.0 if on_edge else -1.0

        if self.pos in self.visited:
            reward -= 0.5
        self.visited.add(self.pos)

        return self._get_state(), reward, False

## Deep Derivation: Level Set Methods and Curvature Motion

### Step 1: Level Set Representation of Boundaries

Instead of parameterizing a contour $\mathbf{v}(s)$, embed it as the **zero level set** of a function $\phi: \mathbb{R}^2 \to \mathbb{R}$:

$$\Gamma = \{(x,y) : \phi(x,y) = 0\}$$

with the convention: $\phi > 0$ inside, $\phi < 0$ outside.

**Advantage:** Topology changes (splitting, merging) are handled automatically — no re-parameterization needed.

### Step 2: Hamilton-Jacobi Evolution Equation

The contour evolves according to:

$$\frac{\partial \phi}{\partial t} + F |\nabla \phi| = 0$$

where $F$ is the **speed function** (positive = expand, negative = shrink).

**Proof this moves the zero level set at speed $F$:**

For a point $\mathbf{x}(t)$ on the zero level set: $\phi(\mathbf{x}(t), t) = 0$. Differentiating:

$$\nabla \phi \cdot \frac{d\mathbf{x}}{dt} + \frac{\partial \phi}{\partial t} = 0$$

The normal velocity is $v_n = \frac{d\mathbf{x}}{dt} \cdot \mathbf{n} = -\frac{\partial \phi / \partial t}{|\nabla \phi|}$

From the evolution equation: $v_n = F$. $\blacksquare$

### Step 3: Curvature of the Level Set

The mean curvature of the zero level set is:

$$\kappa = \text{div}\left(\frac{\nabla \phi}{|\nabla \phi|}\right) = \frac{\phi_{xx}\phi_y^2 - 2\phi_{xy}\phi_x\phi_y + \phi_{yy}\phi_x^2}{(\phi_x^2 + \phi_y^2)^{3/2}}$$

**Curvature-driven motion** uses $F = \kappa$:

$$\frac{\partial \phi}{\partial t} = \kappa |\nabla \phi| = \text{div}\left(\frac{\nabla \phi}{|\nabla \phi|}\right) |\nabla \phi|$$

This smooths the contour (shrinks convex parts, expands concave parts).

### Step 4: Geodesic Active Contour

Combining edge attraction with curvature regularization:

$$\frac{\partial \phi}{\partial t} = g(I) \left(\kappa + \nu\right) |\nabla \phi| + \nabla g \cdot \nabla \phi$$

where:
- $g(I) = \frac{1}{1 + |\nabla(G_\sigma * I)|^2}$ is the **edge indicator** ($g \approx 0$ at edges, $g \approx 1$ in smooth regions)
- $\nu$ is a constant **balloon force** (encourages expansion/contraction)
- $\nabla g \cdot \nabla \phi$ attracts the contour toward edges

**Proof the contour stops at edges:**

At an edge, $g(I) \approx 0$, so the curvature and balloon forces vanish. Only $\nabla g \cdot \nabla \phi$ remains, which pushes the contour toward the edge center. At equilibrium, $\nabla g \cdot \nabla \phi = 0$ (contour aligned with edge). $\blacksquare$

### Step 5: Signed Distance Function Reinitialization

For numerical stability, $\phi$ should remain a **signed distance function** ($|\nabla \phi| = 1$). After evolution, reinitialize by solving:

$$\frac{\partial \phi}{\partial \tau} = \text{sign}(\phi_0)(1 - |\nabla \phi|)$$

This PDE evolves $\phi$ (in artificial time $\tau$) until $|\nabla \phi| = 1$ everywhere, preserving the zero level set.

### Step 6: RL Agent vs Level Set Methods

| Property | Level Set | RL Boundary Agent |
|----------|----------|------------------|
| Topology handling | Automatic | Implicit (multiple paths) |
| Speed function | Hand-designed | Learned from data |
| Convergence | Gradient descent | Q-learning exploration |
| Real-time | Slow (PDE solver) | Fast (lookup table) |

The RL agent effectively learns a **data-driven speed function** through the Q-values, adapting to image-specific edge characteristics rather than relying on generic gradient-based forces.

---
## 4. Q-Learning Boundary Agent

In [ ]:
class BoundaryAgent:
    def __init__(self, num_actions=8, alpha=0.15, gamma=0.9, epsilon=1.0, eps_decay=0.998, eps_min=0.05):
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.eps_decay = eps_decay
        self.eps_min = eps_min
        self.q_table = defaultdict(lambda: np.zeros(num_actions))

    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.num_actions)
        return int(np.argmax(self.q_table[state]))

    def update(self, state, action, reward, next_state, done):
        target = reward if done else reward + self.gamma * np.max(self.q_table[next_state])
        self.q_table[state][action] += self.alpha * (target - self.q_table[state][action])

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

---
## 5. Training

In [ ]:
def train_boundary_agent(image, edge_map, boundary_mask, num_episodes=300, max_steps=250):
    env = BoundaryTracingEnv(image, edge_map, boundary_mask, max_steps=max_steps)
    agent = BoundaryAgent(num_actions=8, alpha=0.15, gamma=0.9, epsilon=1.0, eps_decay=0.99, eps_min=0.05)

    history = {'episode': [], 'reward': [], 'on_edge_pct': [], 'path_length': [], 'loop_closed': []}
    saved_paths = {}

    for ep in range(num_episodes):
        state = env.reset()
        total_reward = 0.0

        for step in range(max_steps):
            action = agent.select_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward
            if done:
                break

        agent.decay_epsilon()

        on_edge = sum(1 for r, c in env.path if boundary_mask[r, c])
        on_edge_pct = on_edge / max(len(env.path), 1)
        loop_closed = (abs(env.pos[0]-env.start_pos[0]) + abs(env.pos[1]-env.start_pos[1])) <= 1 and env.steps > 10

        history['episode'].append(ep)
        history['reward'].append(total_reward)
        history['on_edge_pct'].append(on_edge_pct)
        history['path_length'].append(len(env.path))
        history['loop_closed'].append(loop_closed)

        if ep in [0, 49, 99, 149, 199, 249, 299]:
            saved_paths[ep] = (env.path.copy(), env.start_pos, loop_closed)

        if ep % 50 == 0:
            print(f'Ep {ep:3d} | Reward: {total_reward:7.1f} | On-edge: {on_edge_pct:.2%} | Length: {len(env.path):3d} | Loop: {loop_closed} | ε: {agent.epsilon:.3f}')

    return agent, history, saved_paths

agent, history, saved_paths = train_boundary_agent(image, edge_map, boundary_mask, num_episodes=300)
print('\nTraining complete.')

## Deep Derivation: Boundary Detection as Graph Search and Shortest Path

### Step 1: Boundary Detection as Shortest Path Problem

Define a weighted graph $G = (V, E, w)$ where:
- $V$: pixels in the image
- $E$: edges between adjacent pixels (8-connected)
- $w(p,q) = 1 - g(p,q)$ where $g$ is the edge strength between $p$ and $q$

The optimal boundary between two points is the **shortest path** in this graph:

$$\Gamma^* = \arg\min_{\Gamma} \sum_{(p_i, p_{i+1}) \in \Gamma} w(p_i, p_{i+1})$$

### Step 2: Dijkstra's Algorithm Complexity

Dijkstra finds the shortest path in $O(|V| \log |V| + |E|)$ time. For an $N \times N$ image: $O(N^2 \log N)$.

**Proof of optimality (relaxation):**

When vertex $u$ is extracted from the priority queue, $d[u] = \delta(s, u)$ (true shortest distance). Suppose not: then $\exists$ vertex $v$ on the shortest path $s \to u$ still in the queue with $d[v] < d[u]$. But $v$ would be extracted before $u$ — contradiction. $\blacksquare$

### Step 3: Live Wire (Intelligent Scissors)

The Live Wire method uses Dijkstra with cost:

$$w(p, q) = w_z f_z(q) + w_d f_d(p, q) + w_g f_g(q)$$

where:
- $f_z$: zero-crossing of Laplacian (edge location)
- $f_d$: gradient direction mismatch (boundary smoothness)
- $f_g = 1 - \frac{|\nabla I(q)|}{\max |\nabla I|}$ (gradient magnitude — prefer strong edges)

### Step 4: Connection to RL Policy

Our RL agent learns an **implicit cost function** through the Q-values. The greedy policy $\pi(s) = \arg\max_a Q(s,a)$ at each step chooses the direction that maximizes expected return.

**Theorem:** Under optimal Q-values, the greedy RL policy traces the path maximizing cumulative reward. If reward = edge strength, this is equivalent to finding the **maximum-weight path** in the edge graph — the dual of Dijkstra's shortest path.

$$\pi^* = \arg\max_\pi \sum_{t=0}^T \gamma^t r_t \iff \Gamma^* = \arg\max_\Gamma \sum_{i} \gamma^i g(p_i) \quad \blacksquare$$

### Step 5: Hausdorff Distance for Boundary Evaluation

The Hausdorff distance between predicted boundary $P$ and ground truth $G$:

$$d_H(P, G) = \max\left\{\sup_{p \in P} \inf_{g \in G} \|p - g\|, \; \sup_{g \in G} \inf_{p \in P} \|g - p\|\right\}$$

**Properties:**
1. $d_H(P, G) = 0 \iff P = G$ (identity)
2. $d_H(P, G) = d_H(G, P)$ (symmetry)
3. $d_H(P, Q) \leq d_H(P, G) + d_H(G, Q)$ (triangle inequality)

The Hausdorff distance is a **metric** — it measures the worst-case boundary deviation, complementing the average-case BF-score used in training.

---
## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

window = 20
def smooth(arr, w=window):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

axes[0].plot(smooth(history['reward']), color='teal', linewidth=1.5)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Total Reward'); axes[0].set_title('Reward (smoothed)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(smooth(history['on_edge_pct']), color='coral', linewidth=1.5)
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('On-Edge %'); axes[1].set_title('Fraction of Path on Boundary')
axes[1].grid(True, alpha=0.3)

axes[2].plot(smooth(history['path_length']), color='purple', linewidth=1.5)
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Path Length'); axes[2].set_title('Path Length')
axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 7. Agent Path Visualization

In [ ]:
snap_keys = sorted(saved_paths.keys())
n_plots = len(snap_keys)
fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

for idx, ep in enumerate(snap_keys):
    path, start, looped = saved_paths[ep]
    ax = axes[idx]
    ax.imshow(edge_map, cmap='gray', alpha=0.7)

    path_arr = np.array(path)
    ax.plot(path_arr[:, 1], path_arr[:, 0], 'r-', linewidth=1, alpha=0.8)
    ax.plot(start[1], start[0], 'go', markersize=8, label='Start')
    ax.plot(path_arr[-1, 1], path_arr[-1, 0], 'bs', markersize=8, label='End')

    status = 'Loop closed' if looped else 'Open'
    on_edge = sum(1 for r, c in path if boundary_mask[r, c]) / max(len(path), 1)
    ax.set_title(f'Ep {ep} ({status})\nOn-edge: {on_edge:.0%}', fontsize=10)
    ax.legend(fontsize=7, loc='lower right')
    ax.axis('off')

plt.suptitle('Agent Boundary Tracing Over Training', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Deep Derivation: Directional Policies and Circular Statistics for Boundary Tracing

### Step 1: Circular Action Distribution Analysis

Since our actions are 8 compass directions, the action distribution is **circular** data. Standard statistics (mean, variance) don't apply directly.

**Circular mean direction:**

$$\bar{\theta} = \text{atan2}\left(\sum_{i} \sin \theta_i, \sum_{i} \cos \theta_i\right)$$

where $\theta_i = \frac{2\pi a_i}{8}$ for action $a_i \in \{0, \ldots, 7\}$.

**Circular variance (Mardia):**

$$V = 1 - \bar{R}, \quad \bar{R} = \frac{1}{N}\sqrt{\left(\sum_i \cos\theta_i\right)^2 + \left(\sum_i \sin\theta_i\right)^2}$$

$\bar{R} \in [0, 1]$: concentration parameter. $\bar{R} = 1$ means all actions are the same direction (degenerate policy). $\bar{R} = 0$ means uniformly distributed (random exploration).

### Step 2: Von Mises Distribution as Directional Policy Prior

The von Mises distribution is the circular analog of the Gaussian:

$$f(\theta; \mu, \kappa) = \frac{e^{\kappa \cos(\theta - \mu)}}{2\pi I_0(\kappa)}$$

where $\mu$ is the mean direction and $\kappa$ is the concentration parameter, and $I_0$ is the modified Bessel function of order 0.

**Proof $\kappa \to 0$ gives uniform, $\kappa \to \infty$ gives point mass:**

For $\kappa = 0$: $f(\theta) = \frac{1}{2\pi I_0(0)} = \frac{1}{2\pi}$ (uniform).

For $\kappa \to \infty$: $e^{\kappa\cos(\theta-\mu)}$ is sharply peaked at $\theta = \mu$. $\blacksquare$

### Step 3: Optimal Boundary Direction as Gradient Perpendicular

The true boundary direction at pixel $(x, y)$ is perpendicular to the image gradient:

$$\theta_{\text{boundary}} = \arctan\frac{\partial I / \partial y}{\partial I / \partial x} + \frac{\pi}{2}$$

**Proof:** The gradient $\nabla I$ points in the direction of maximum intensity change (across the edge). The boundary follows the edge, which is perpendicular to this direction. $\blacksquare$

**RL connection:** The learned Q-values implicitly encode this relationship. The agent learns that the best action at each state is the one closest to $\theta_{\text{boundary}}$:

$$\pi^*(s) \approx \arg\min_a |\theta_a - \theta_{\text{boundary}}(s)|$$

### Step 4: Path Smoothness as Angular Momentum

Define angular change between consecutive steps:

$$\Delta\theta_t = \theta_{a_t} - \theta_{a_{t-1}} \pmod{2\pi}$$

**Smooth paths** have small $|\Delta\theta_t|$ (low angular acceleration). The agent's learned policy should exhibit:

$$\mathbb{E}[|\Delta\theta_t|] \ll \pi \quad \text{(prefers continuing in similar direction)}$$

This emerges naturally from training: sharp turns typically move the agent off-edge (negative reward), while smooth curves follow the boundary (positive reward).

**Mean path curvature:**

$$\bar{\kappa}_{\text{path}} = \frac{1}{T}\sum_{t=1}^{T} \frac{|\Delta\theta_t|}{\|\mathbf{p}_t - \mathbf{p}_{t-1}\|} \approx \frac{1}{T}\sum_{t=1}^T |\Delta\theta_t|$$

since step lengths are constant ($= 1$ pixel for cardinal, $= \sqrt{2}$ for diagonal). $\blacksquare$

---
## 8. Detailed Path Analysis on the Best Episode

In [ ]:
env_eval = BoundaryTracingEnv(image, edge_map, boundary_mask, max_steps=300)
state = env_eval.reset()
for step in range(300):
    action = int(np.argmax(agent.q_table[state]))
    state, _, done = env_eval.step(action)
    if done:
        break

path = env_eval.path
path_arr = np.array(path)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Path on image
axes[0].imshow(image, cmap='gray')
axes[0].plot(path_arr[:, 1], path_arr[:, 0], 'r-', linewidth=1.5, alpha=0.9)
axes[0].plot(path_arr[0, 1], path_arr[0, 0], 'go', markersize=10)
axes[0].plot(path_arr[-1, 1], path_arr[-1, 0], 'bs', markersize=10)
axes[0].set_title('Agent Path on Image'); axes[0].axis('off')

# Path on edge map
axes[1].imshow(edge_map, cmap='hot')
axes[1].plot(path_arr[:, 1], path_arr[:, 0], 'cyan', linewidth=1.5, alpha=0.9)
axes[1].set_title('Path on Edge Map'); axes[1].axis('off')

# On-edge vs off-edge pixels colour-coded
axes[2].imshow(image, cmap='gray', alpha=0.4)
for r, c in path:
    color = 'lime' if boundary_mask[r, c] else 'red'
    axes[2].plot(c, r, '.', color=color, markersize=2)
on_patch = mpatches.Patch(color='lime', label='On edge')
off_patch = mpatches.Patch(color='red', label='Off edge')
axes[2].legend(handles=[on_patch, off_patch], fontsize=9)
axes[2].set_title('Edge Classification of Path'); axes[2].axis('off')

plt.suptitle('Greedy-Policy Path Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

on_edge_count = sum(1 for r, c in path if boundary_mask[r, c])
print(f'Path length: {len(path)}')
print(f'On-edge pixels: {on_edge_count} / {len(path)} = {on_edge_count/len(path):.1%}')
dist = abs(path[-1][0]-path[0][0]) + abs(path[-1][1]-path[0][1])
print(f'Distance start→end: {dist} (loop closed: {dist <= 1 and len(path) > 10})')

---
## 9. Action Distribution Analysis

In [ ]:
action_counts = np.zeros(8)
env_test = BoundaryTracingEnv(image, edge_map, boundary_mask, max_steps=300)
for trial in range(50):
    state = env_test.reset()
    for step in range(300):
        action = int(np.argmax(agent.q_table[state]))
        action_counts[action] += 1
        state, _, done = env_test.step(action)
        if done:
            break

action_counts /= action_counts.sum()

fig, ax = plt.subplots(figsize=(8, 5), subplot_kw={'projection': 'polar'})
angles = np.linspace(0, 2*np.pi, 8, endpoint=False)
angles_closed = np.append(angles, angles[0])
counts_closed = np.append(action_counts, action_counts[0])

ax.plot(angles_closed, counts_closed, 'o-', color='teal', linewidth=2)
ax.fill(angles_closed, counts_closed, alpha=0.25, color='teal')
ax.set_xticks(angles)
ax.set_xticklabels(DIR_NAMES)
ax.set_title('Greedy Policy Action Distribution', pad=20, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Summary

In this notebook we:

1. **Formulated boundary tracing as an MDP** with 8 directional actions and edge-proximity rewards.
2. **Designed a compact state** using quantised local gradient values and direction history.
3. **Trained a tabular Q-learning agent** that learns to follow object boundaries.
4. **Visualized the agent's path** overlaid on the image, showing how it traces object contours.
5. **Analysed action distributions** to understand learned directional preferences.

### Key Takeaways

| Concept | Detail |
|---|---|
| State | Local $3{\times}3$ gradient patch + last direction |
| Actions | 8 compass directions (N, NE, …, NW) |
| Reward | $+1$ on edge, $-1$ off edge, $+5$ loop bonus |
| Algorithm | Tabular Q-learning |
| Metric | On-edge fraction, path length, loop closure |

**Next →** Notebook 04 tackles **Semantic Segmentation with Deep RL**.